Setup

In [1]:
import os, shutil, subprocess, sys, torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} | {props.total_memory / 1e9:.2f} GB")

for path in ["/kaggle/working", "/kaggle/tmp", "/tmp"]:
    if os.path.exists(path):
        total, used, free = shutil.disk_usage(path)
        print(f"Disk {path}: {free/1e9:.2f} GB free / {total/1e9:.2f} GB total")

java_check = subprocess.run(["java", "-version"], capture_output=True, text=True)
print("Java status:", "OK" if java_check.returncode == 0 else "MISSING")
if java_check.stderr:
    print(" ", java_check.stderr.splitlines()[0])

Python: 3.13.16
PyTorch: 2.11.0+cu130 | CUDA available: True
GPU count: 2
  GPU 0: Tesla T4 | 15.64 GB
  GPU 1: Tesla T4 | 15.64 GB
Disk /kaggle/working: 20.94 GB free / 20.96 GB total
Disk /tmp: 2700.00 GB free / 8758.39 GB total
Java status: OK
  openjdk version "21.0.12.1" 2026-08-18


In [2]:
import os

os.makedirs("/kaggle/tmp/hf_cache", exist_ok=True)
os.environ["HF_HOME"] = "/kaggle/tmp/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

!pip install -q -U peft accelerate huggingface_hub wandb pycocoevalcap easyocr rapidocr onnxruntime open-clip-torch

import transformers, peft, accelerate
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
for name in ["HF_TOKEN", "WANDB_API_KEY"]:
    try:
        val = secrets.get_secret(name)
        print(f"Secret {name}: FOUND (len={len(val)})")
    except Exception as e:
        print(f"Secret {name}: MISSING ({type(e).__name__})")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.109.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 2.4.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
transformers: 5.20.2
peft: 0.22.0
accelerate: 1.16.0
Secret HF_TOKEN: MISSING (BackendError)
Secret WANDB_API_KEY: MISSING (BackendError)


BLIP

In [3]:
import time
import torch
from PIL import Image, ImageDraw, ImageFont
from transformers import Blip2Processor, Blip2ForConditionalGeneration

# Create a crisp synthetic test image with visible text (no network dependency)
img = Image.new("RGB", (512, 512), color=(240, 235, 225))
draw = ImageDraw.Draw(img)
draw.rectangle([120, 90, 392, 430], fill=(200, 45, 45), outline=(40, 40, 40), width=4)
draw.rectangle([145, 160, 367, 320], fill=(255, 255, 255))
font = ImageFont.load_default()
draw.text((165, 190), "ORGANIC\nTOMATO SOUP\nNET WT 14 OZ", fill=(20, 20, 20), font=font)

model_id = "Salesforce/blip2-opt-2.7b"
t0 = time.time()
processor = Blip2Processor.from_pretrained(model_id)
model = Blip2ForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map={"": 0},
)
model.eval()
print(f"Model loaded in {time.time() - t0:.1f}s")
print(f"Allocated after load: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")

prompt = "Visible text: ORGANIC TOMATO SOUP NET WT 14 OZ. Alt-text:"
inputs = processor(images=img, text=prompt, return_tensors="pt").to("cuda:0", torch.float16)

print("input_ids shape:", inputs["input_ids"].shape)
print("decoded input_ids:", repr(processor.tokenizer.decode(inputs["input_ids"][0])))

torch.cuda.reset_peak_memory_stats(0)
t1 = time.time()
with torch.no_grad():
    out = model.generate(
        **inputs,
        num_beams=5,
        max_new_tokens=60,
        no_repeat_ngram_size=3,
    )
elapsed = time.time() - t1

full_text = processor.batch_decode(out, skip_special_tokens=True)[0]
new_tokens = out.shape[1] - inputs["input_ids"].shape[1]
print(f"Generated output shape: {out.shape} | Time: {elapsed:.2f}s | Peak VRAM: {torch.cuda.max_memory_allocated(0)/1e9:.2f} GB")
print("Raw decoded output:", repr(full_text))

processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.03k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/882 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

... (3 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

Model loaded in 119.5s
Allocated after load: 7.70 GB
input_ids shape: torch.Size([1, 55])
decoded input_ids: '<image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image></s>Visible text: ORGANIC TOMATO SOUP NET WT 14 OZ. Alt-text:'
Generated output shape: torch.Size([1, 75]) | Time: 6.17s | Peak VRAM: 7.85 GB
Raw decoded output: 'Visible text: ORGANIC TOMATO SOUP NET WT 14 OZ. Alt-text: This soup is vegan, gluten-free, non-GMO, and made with organic ingredients\n'
